In [4]:
%%writefile practice10_4.cpp
// Анализ масштабируемости распределённой программы (MPI)
// измерить время выполнения при различном числе процессов;
// оценить strong scaling и weak scaling;
// проанализировать влияние коммуникационных операций (MPI_Reduce, MPI_Allreduce);
// сделать вывод о масштабируемости алгоритма и его практических ограничениях.
#include <mpi.h>        // Библиотека MPI для распределённых вычислений
#include <iostream>     // Для вывода результатов
#include <vector>       // Для динамических массивов
#include <cstdlib>      // Для генерации случайных чисел

#define N 1000000       // Размер массива (1 млн элементов)

int main(int argc, char* argv[]) {
    MPI_Init(&argc, &argv);                        // Инициализация MPI среды, обязательная для всех MPI программ

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);          // Получаем уникальный номер процесса (rank)
    MPI_Comm_size(MPI_COMM_WORLD, &size);          // Получаем общее число процессов

    int local_N = N / size;                        // Количество элементов для каждого процесса
    int remainder = N % size;                      // Остаток, если N не делится на size
    if (rank == size - 1) local_N += remainder;    // Последний процесс получает остаток

    std::vector<int> local_data(local_N);          // Локальный массив на каждом процессе

    srand(rank + 1);                               // Инициализация генератора случайных чисел разным seed для каждого процесса
    for (int i = 0; i < local_N; i++)
        local_data[i] = rand() % 100;             // Заполнение массива случайными числами от 0 до 99

    double start_time = MPI_Wtime();               // Замер времени начала выполнения на этом процессе

    long long local_sum = 0;                       // Локальная сумма элементов
    int local_min = local_data[0];                 // Локальный минимум
    int local_max = local_data[0];                 // Локальный максимум

    for (int i = 0; i < local_N; i++) {           // Проходим по всем локальным элементам
        local_sum += local_data[i];               // Считаем локальную сумму
        if (local_data[i] < local_min) local_min = local_data[i]; // Обновляем локальный минимум
        if (local_data[i] > local_max) local_max = local_data[i]; // Обновляем локальный максимум
    }

    long long global_sum = 0;                      // Глобальная сумма всех элементов
    int global_min = 0;                            // Глобальный минимум
    int global_max = 0;                            // Глобальный максимум

    MPI_Reduce(&local_sum, &global_sum, 1, MPI_LONG_LONG, MPI_SUM, 0, MPI_COMM_WORLD); // Суммирование локальных сумм всех процессов, результат у процесса 0
    MPI_Reduce(&local_min, &global_min, 1, MPI_INT, MPI_MIN, 0, MPI_COMM_WORLD);       // Поиск глобального минимума
    MPI_Reduce(&local_max, &global_max, 1, MPI_INT, MPI_MAX, 0, MPI_COMM_WORLD);       // Поиск глобального максимума

    double end_time = MPI_Wtime();                 // Замер времени окончания выполнения
    double elapsed = end_time - start_time;       // Вычисление затраченного времени на этом процессе

    if (rank == 0) {                              // Вывод результатов только процессом 0
        std::cout << "Глобальная сумма: " << global_sum << std::endl;   // Печать глобальной суммы
        std::cout << "Глобальный минимум: " << global_min << std::endl; // Печать глобального минимума
        std::cout << "Глобальный максимум: " << global_max << std::endl;// Печать глобального максимума
        std::cout << "Время выполнения программы: " << elapsed << " секунд" << std::endl; // Время выполнения
    }

    MPI_Finalize();                               // Завершение работы MPI
    return 0;                                     // Завершение программы
}



Overwriting practice10_4.cpp


In [5]:
# Компиляция
!mpic++ practice10_4.cpp -o practice10_4
# mpirun — утилита, которая запускает MPI-программу на указанном числе процессов
# --allow-run-as-root — разрешает запуск MPI от root (Colab запускает ядра как root)
# --oversubscribe — игнорирует количество доступных виртуальных CPU, позволяя запускать больше процессов, чем физически есть
# -np 2 — число процессов MPI (2 процесса)

# Запуск с 2 процессами
!mpirun --allow-run-as-root --oversubscribe -np 2 ./practice10_4
# Запуск с 8 процессами
!mpirun --allow-run-as-root --oversubscribe -np 8 ./practice10_4
# Запуск с 16 процессами
!mpirun --allow-run-as-root --oversubscribe -np 16 ./practice10_4

Глобальная сумма: 49535586
Глобальный минимум: 0
Глобальный максимум: 99
Время выполнения программы: 0.010718 секунд
Глобальная сумма: 49531299
Глобальный минимум: 0
Глобальный максимум: 99
Время выполнения программы: 0.00716848 секунд
Глобальная сумма: 49546353
Глобальный минимум: 0
Глобальный максимум: 99
Время выполнения программы: 0.0028771 секунд
